In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 48.06it/s]


Numba compilation complete!


In [2]:
#initial file processing
labcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = labcomp


filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
savedir = titledpath + filedir + "Compilation with delta\\2025deltagcollection\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ACR"
respondercsv = responder + ".csv"
wt = "w1118"


In [3]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

for file_no in os.listdir(openPath): 
    if respondercsv in file_no and "w1118" not in file_no :   
        f = os.path.join(openPath, file_no)
        dfe=pd.read_csv(f)
        exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
        driver = file_no.split(" ")[0]
        lstnew.append(driver)
lst = lstnew.copy()

#processing ONLY specific names
#lst = ["MB112C"]

print(lst)

['MB011B', 'MB018B', 'MB027B', 'MB057B', 'MB077B', 'MB080C', 'MB082C', 'MB083C', 'MB093C', 'MB112C', 'MB210B', 'MB242A', 'MB310C', 'MB319C', 'MB323B', 'MB399B', 'MB434B', 'MB542B', 'MB543B', 'R58', 'R76B09', 'SS01127', 'SS01188', 'SS01298', 'SS01308', 'SS01337', 'SS01388', 'SS46348', 'SS52050', 'SS67662', 'SS67727', 'SS67741', 'SS75199', 'SS75200', 'SS76094', 'SS77383', 'SS77424', 'SS77442', 'SS77450', 'SS80896', 'SS80958', 'SS81353', 'SS81521', 'SS86947', 'SS95118', 'SS97567', 'Th-Gal4', 'VT999036']


In [20]:
diff = pd.DataFrame()
diffbs = pd.DataFrame()

for n in lst:
    driver = n
    print(n)
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    #adjust this depending on timeframe
    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    #mean_diff
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True)

    #delta_g    
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    #df_t = NLMATH.timetype(dfwt, dfexpt).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    #df_d = pd.concat([NLMATH.totaldisp(dfexpt, "Expt"), NLMATH.totaldisp(dfwt, "WT")]).reset_index(drop=True)
    #df_bp = NLMATH.bheight(NLMATH.boutheight(dfexpt), NLMATH.boutheight(dfwt)).reset_index(drop=True)
    df_pp = NLMATH.bheight(NLMATH.pauseheight(dfexpt), NLMATH.pauseheight(dfwt)).reset_index(drop=True)
    #df_dispp = pd.concat([NLMATH.displacementbetweenpauses(dfexpt, "Expt"), NLMATH.displacementbetweenpauses(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    df_sim = pd.concat([NLMATH.straightnessindexmeter(dfexpt, "Expt"), NLMATH.straightnessindexmeter(dfwt, "WT")], axis = 0).reset_index(drop=True)

    #ascentdescent
    # updown_df = pd.DataFrame()
    # updown_df = pd.concat([NLMATH.positional_arguments(dfexpt, driver), NLMATH.positional_arguments(dfwt, "w1118")], axis = 0)
    # descendingdf = updown_df[updown_df['index'].str.contains("Descending.*")].reset_index(drop=True)
    # ascendingdf = updown_df[updown_df['index'].str.contains("Ascending.*")].reset_index(drop=True)
    
    #pause and bouts
    wttotalmeanevent, wttotalnumberevent = NLMATH.pausecomp(dfwt, wt)
    expttotalmeanevent, expttotalnumberevent = NLMATH.pausecomp(dfexpt, driver)
        
    #alltgtmeandf_pause = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Pauses"), NLMATH.pausenumber(expttotalmeanevent, n, "Pauses")], axis = 0).reset_index(drop=True)
    alltgtmeandf_bout = pd.concat([NLMATH.pausenumber(wttotalmeanevent, n, "Bouts"), NLMATH.pausenumber(expttotalmeanevent, n, "Bouts")], axis = 0).reset_index(drop=True)
    
    #alltgtnumberdf_pause = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Pauses"), NLMATH.pausenumber(expttotalnumberevent, n, "Pauses")], axis = 0).reset_index(drop=True)
    alltgtnumberdf_bout = pd.concat([NLMATH.pausenumber(wttotalnumberevent, n, "Bouts"), NLMATH.pausenumber(expttotalnumberevent, n, "Bouts")], axis = 0).reset_index(drop=True)
            
    #___________________________________________#    
    # meandiff plots -- you run mean_diff instead of delta_g because since all the binary data is at the same dimension, no standardization is required and empirical delta delta is sufficient
    #dff2_prop = NLMATH.deltaversion_meandiff(df_f, "binary_fallvalue", "fallprop")
    dff2_number = NLMATH.deltaversion_meandiff(df_f, "Fall", "fallnumber")   #number of falls is under deltaversion_binary because the number of flies that fall could actually be so few in number, that the SD is 0, and thus hedges g will not be able to perform since the divisor ==0

    #meandiff plots
    dfs2 = NLMATH.deltaversion_deltag(df_sp, "Velocity", "speed")
    #dft2 = NLMATH.deltaversion_deltag(df_t, "Time", "time") #time spent above 3/4 of height
    dfh2 = NLMATH.deltaversion_deltag(df_h, "Y", "height")
    #dfd2 = NLMATH.deltaversion_deltag(df_d, "displacement", "displacement")
    dfbs2 = NLMATH.deltaversion_deltag(df_bsp, "BSpeed", "bspeed")
    #dfbp2 = NLMATH.deltaversion_deltag(df_bp, "Height", "boutpos")
    dfpp2 = NLMATH.deltaversion_deltag(df_pp, "Height", "pausepos")
    #dfdbp2 = NLMATH.deltaversion_deltag(df_dispp, "avgdisplacementbetweenpause", "displacementbetweenpause")
    dfmv2 = NLMATH.deltaversion_deltag(df_maxv, "maxvelocity", "maxvelocity")
    dfsim2 = NLMATH.deltaversion_deltag(df_sim, "averagestraightnessindex", "straightindex")
    # dfasc = NLMATH.deltaversion_deltag(ascendingdf, "Position", "ascent")
    # dfdesc = NLMATH.deltaversion_deltag(descendingdf, "Position", "descent")
    #pause and bouts
    #dfmp2 = NLMATH.deltaversion_deltag(alltgtmeandf_pause, "Pauses", "meanpause")
    dfmb2 = NLMATH.deltaversion_deltag(alltgtmeandf_bout, "Bouts", "meanbout")     
    #dfnp2 = NLMATH.deltaversion_deltag(alltgtnumberdf_pause, "Pauses", "pause")
    dfnb2 = NLMATH.deltaversion_deltag(alltgtnumberdf_bout, "Bouts", "bout")
    
    
    dftotal = pd.concat([dff2_number, dfs2, dfh2, dfbs2, dfpp2, dfmv2, dfsim2, dfmb2, dfnb2], axis = 1)
    #dftotal = pd.concat([dff2_prop, dff2_number, dfs2, dft2, dfh2, dfd2, dfbs2, dfpp2, dfdbp2, dfmv2, dfasc, dfdesc, dfmp2, dfmb2, dfnp2, dfnb2], axis = 1)
    dftotal['MBON'] = n


    dftotal.set_index("MBON", inplace = True)
    dftotal.to_csv(savedir + n + " x " + responder + "_deltag_allstats.csv")

#2025 collection will have a mix of delta g and mean diff
#20241014 is all solely mean diffs
        


MB011B
MB018B
MB027B
MB057B
MB077B
MB080C
MB082C
MB083C
MB093C
MB112C
MB210B
MB242A
MB310C
MB319C
MB323B
MB399B
MB434B
MB542B
MB543B
R58
R76B09
SS01127
SS01188
SS01298
SS01308
SS01337
SS01388
SS46348
SS52050
SS67662
SS67727
SS67741
SS75199
SS75200
SS76094
SS77383
SS77424
SS77442
SS77450
SS80896
SS80958
SS81353
SS81521
SS86947
SS95118
SS97567
Th-Gal4
VT999036


In [ ]:
df_sp_combined


In [4]:
# Calculate log2(speed in light/speed in dark) for MB112C x ACR
import numpy as np

# Process MB112C x ACR specifically
driver = "MB112C"
responder = 'ACR'
transgenic = driver + " x " + responder
filename = openPath + transgenic + ".csv"
filenamewt = openPath + wt+"_"+ transgenic + ".csv"

# Load data
dfe = pd.read_csv(filename)
dfw = pd.read_csv(filenamewt)

exptdf = dfe.drop(dfe.columns[[0]], axis=1)
wtdf = dfw.drop(dfw.columns[[0]], axis=1)

# Process data
dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))



In [47]:
NLMATH.boutspeed(dfexpt)


,Seconds,ExperimentState,MB112C BSpeed_1,MB112C BSpeed_2,MB112C BSpeed_3,MB112C BSpeed_5,MB112C BSpeed_6,MB112C BSpeed_7,MB112C BSpeed_8,MB112C BSpeed_9,...,MB112C BSpeed_121,MB112C BSpeed_123,MB112C BSpeed_124,MB112C BSpeed_125,MB112C BSpeed_126,MB112C BSpeed_128,MB112C BSpeed_130,MB112C BSpeed_131,MB112C BSpeed_132,MB112C BSpeed_133
0,3.0,Dark,6.924140,14.644529,19.939688,13.606518,19.582014,20.290750,2.483371,11.132336,...,9.121873,5.865006,NaN,7.445113,4.164063,7.086427,10.531068,6.763798,NaN,6.549554
1,3.2,Dark,9.013809,NaN,22.448199,13.761490,18.842585,19.520095,NaN,14.965216,...,10.682061,5.331573,NaN,8.603029,2.507233,7.711069,5.799214,10.908711,NaN,6.579510
2,3.4,Dark,5.803017,6.755121,20.963719,19.244894,13.006433,15.777385,4.322930,19.927456,...,9.683516,3.784515,NaN,6.410138,2.556767,6.796299,7.749952,12.337299,NaN,NaN
3,3.6,Dark,8.842613,9.727064,15.694434,9.587457,10.923275,12.028862,2.786716,14.400197,...,10.721951,6.521157,3.132741,7.914281,NaN,4.865495,8.938726,9.401600,3.737482,7.035419
4,3.8,Dark,4.475506,13.539937,15.161069,NaN,11.048170,NaN,NaN,18.170966,...,10.501078,2.928994,2.496251,3.715112,4.554700,7.752039,NaN,9.948876,2.203136,5.080590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,65.0,Recovery,NaN,4.653963,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.218399,NaN,NaN
288,65.2,Recovery,NaN,6.072412,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
289,65.4,Recovery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
290,65.6,Recovery,NaN,7.398075,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)

,index,BSpeed,ExperimentState,Type,genre
0,MB112C BSpeed_1,7.316164,Dark,Expt,Dark Expt
1,MB112C BSpeed_2,7.790729,Dark,Expt,Dark Expt
2,MB112C BSpeed_3,9.024864,Dark,Expt,Dark Expt
3,MB112C BSpeed_5,11.002649,Dark,Expt,Dark Expt
4,MB112C BSpeed_6,8.542150,Dark,Expt,Dark Expt
...,...,...,...,...,...
913,w1118 BSpeed_233,4.381142,Recovery,WT,Recovery WT
914,w1118 BSpeed_234,2.392500,Recovery,WT,Recovery WT
915,w1118 BSpeed_235,6.582057,Recovery,WT,Recovery WT
916,w1118 BSpeed_237,4.173433,Recovery,WT,Recovery WT


In [44]:
def log2speedratio(dfexpt, dfwt):
    import numpy as np
    import pandas as pd
    
    df_sp_expt = NLMATH.velodabest(dfexpt, "Expt", "Velocity")
    df_sp_wt = NLMATH.velodabest(dfwt, "WT", "Velocity")
    total_df = pd.DataFrame()
    for n in [df_sp_expt, df_sp_wt]:
        pivot_df = n.pivot(columns='ExperimentState', values='Velocity')
        pivot_df['Log2 Speed_Ratio'] = np.log2(pivot_df['Full'] / pivot_df['Dark'])
        pivot_df['Type'] = n.groupby(n.index)['Type'].first()
        final_df = pivot_df[['Log2 Speed_Ratio', 'Type']]
        
        total_df = pd.concat([total_df, final_df])
    
    return total_df.reset_index(drop=False)

def singledelta(df, metric, dfnaming):
    import dabest
    import pandas as pd
    
    df_dbsingle = dabest.load(df, idx = ("Expt", "WT"), y = metric, x = 'Type')
    df_singledelta = pd.DataFrame({dfnaming +"_bootstrap": df_dbsingle.mean_diff.results.bootstraps[0].tolist(), dfnaming +"_meandiff": round(float(df_dbsingle.mean_diff.results.difference),3)})
    return df_singledelta